# Tema 21 — Generación de texto a partir de imágenes (Image Captioning con BLIP)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-11/Tema-21/Tema_21.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

En esta práctica usamos un modelo **multimodal** de *imagen → texto*: **BLIP** (*Bootstrapping Language-Image Pre-training*) de Salesforce, a través de la librería `transformers` de Hugging Face. El modelo recibe una **imagen** y genera automáticamente una **descripción en lenguaje natural** (*image captioning*).

## 1. Instalación de dependencias

`BLIP` se descarga desde el *hub* de Hugging Face y se ejecuta con **PyTorch**. En **Google Colab** casi todo viene preinstalado; la siguiente celda instala lo necesario **solo si estás en Colab**. En local, instala las dependencias una vez siguiendo el `README.md` de la carpeta.

In [ ]:
# === Instalación de dependencias (solo en Google Colab) ===
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q transformers torch pillow
    print("Setup de Colab completado.")
else:
    print("Entorno local detectado. Asegúrate de tener: transformers, torch, pillow (ver README.md).")

## 2. Cargar el modelo BLIP

Descargamos el modelo preentrenado `Salesforce/blip-image-captioning-base` y su `processor` (encargado de convertir la imagen al formato que espera el modelo). La **primera** ejecución descarga los pesos (unos cientos de MB) y puede tardar.

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch
import os

modelo = "Salesforce/blip-image-captioning-base"

processor = BlipProcessor.from_pretrained(modelo)
model = BlipForConditionalGeneration.from_pretrained(modelo)

print("Modelo BLIP cargado:", modelo)

## 3. Preparar la imagen de entrada

El modelo necesita una imagen. Puedes:

- Colocar tu propia imagen junto al notebook (por ejemplo `sample.jpg`), o
- En **Colab**, subirla desde el panel **Archivos**, o
- Dejar que la siguiente celda **descargue una imagen de ejemplo** si no encuentra ninguna.

La imagen se abre con `PIL` y se convierte a **RGB** (BLIP espera 3 canales).

In [ ]:
import urllib.request

# La imagen de entrada: usa una local si existe; si no, descarga una de ejemplo.
ruta_imagen = "sample.jpg"

if not os.path.exists(ruta_imagen):
    # Imagen de ejemplo del dataset COCO (usada en la documentación de Hugging Face).
    url_ejemplo = "http://images.cocodataset.org/val2017/000000039769.jpg"
    print("No se encontró una imagen local; descargando una de ejemplo...")
    urllib.request.urlretrieve(url_ejemplo, ruta_imagen)
    print("Imagen de ejemplo guardada como:", ruta_imagen)

imagen = Image.open(ruta_imagen).convert("RGB")
imagen

## 4. Generar la descripción (captioning)

Pasamos la imagen por el `processor` para obtener los tensores de entrada y luego llamamos a `model.generate()` para producir la descripción. `max_new_tokens` limita la longitud del texto generado. Con `torch.no_grad()` desactivamos el cálculo de gradientes (inferencia: más rápida y con menos memoria).

In [ ]:
# Captioning sin prompt (descripción libre de la imagen).
inputs = processor(images=imagen, return_tensors="pt")

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=50)

descripcion = processor.decode(output[0], skip_special_tokens=True)

print("Descripción generada:")
print(descripcion)